# Практикум: сборка сфер SiO₂ в высыхающем водно-спиртовом слое

Этот исполняемый пример дополняет [лекцию 10](10-scientific_and_stochastic_modeling.md). Мы сравним **двумерную локальную упаковку** при двух условных сценариях концентрирования частиц, вычислим парную корреляцию $g(r)$ и шестилучевой порядок $\psi_6$, затем отдельно смоделируем перенос к краю капли. Данные и параметры ниже **учебные**, а не подгонка по конкретному эксперименту.

Запускайте ячейки сверху вниз из чистого ядра. Требуются только `numpy` и `matplotlib` из [`requirements.txt`](../requirements.txt); полный расчёт на обычном компьютере занимает порядка десятков секунд. Фиксированные `seed` позволяют воспроизвести пример, но не заменяют повторения независимых экспериментов.

**Граница модели:** сферические частицы описаны центрами дисков в периодической плоскости. Результат относится к порядку *одного поверхностного слоя*. Ни высокое $|\psi_6|$, ни регулярный рисунок не доказывают образование трёхмерного опала или оптического цвета. Эволюция состава вода–этанол, капиллярные силы, гидродинамические взаимодействия, DLVO-потенциал, деформация подложки и фиксация частиц здесь не вычисляются.


## 1. Паспорт условной системы и единицы

Пользовательские входы: характерный **гидродинамический диаметр** $d_0$ в нм, измеренный $\zeta$-потенциал в мВ, начальная объёмная доля этанола **по исходным объёмам смешиваемых жидкостей**, температура $T$ в K и **отдельно** динамическая вязкость $\eta$ в мПа·с для полученной смеси при этой температуре. Исходные объёмы воды и этанола могут не складываться в конечный объём раствора. Вода–этанол не имеет линейной по составу вязкости; подставлять $\eta$ из одной только доли этанола нельзя. В ходе каждого прогона состав и вязкость считаются постоянными: настоящее высыхание смеси требует как минимум функций $x_{\mathrm{EtOH}}(t)$ и $\eta(t)$.

Для одиночной сферы вдали от стенки используется приближение Стокса–Эйнштейна $D_0=k_BT/(3\pi\eta d_0)$; диаметр включает гидродинамическую оболочку. Приближение следует заново оценить около подложки, при агрегации или высокой концентрации. Для частиц размера $d_i$ полагаем $D_i/D_0=d_0/d_i$ — ещё одно упрощение.

При сохранении **того же безразмерного** расписания сжатия изменение $T$, $\eta$ или общего масштаба $d_0$ меняет перевод координат и времени в физические единицы, но не рисунок упаковки. Если экспериментальная скорость сушки фиксирована *в секундах*, надо заново рассчитать расписание сжатия в единицах $d_0^2/D_0$; состав дополнительно меняет параметры взаимодействия, которые наш пример не выводит из смеси.

Измеренный $\zeta$ **не определяет** однозначно потенциал взаимодействия: нужны ионная сила, pH, диэлектрические свойства смеси, химия поверхности и допущения об электрическом двойном слое. Ниже $\zeta$ управляет только *объявленной сценарной связью* $A(\zeta)=A_{\rm ref}(|\zeta|/\zeta_{\rm ref})^2$ при фиксированных остальных условиях. Это способ проверить чувствительность алгоритма, **не физический закон**, не расчёт DLVO и не прогноз при замене растворителя. Для реального образца эффективные барьер $A$, длину взаимодействия $\lambda$ и адгезию $H$ надо независимо обосновать/калибровать.


In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print(f"Python {sys.version.split()[0]}, NumPy {np.__version__}, Matplotlib {matplotlib.__version__}")

K_B = 1.380649e-23  # Дж/К
SYSTEM = {
    "diameter_nm": 200.0,
    "zeta_mV": -35.0,
    "ethanol_volume_fraction": 0.25,
    "temperature_K": 298.15,
    "viscosity_mPa_s": 2.0,  # условное введённое значение; сверить с измерением смеси
}
assert SYSTEM["diameter_nm"] > 0 and SYSTEM["temperature_K"] > 0
assert SYSTEM["viscosity_mPa_s"] > 0
assert 0 <= SYSTEM["ethanol_volume_fraction"] <= 1

def diffusion_nm2_s(system):
    eta_Pa_s = system["viscosity_mPa_s"] * 1e-3
    diameter_m = system["diameter_nm"] * 1e-9
    return K_B * system["temperature_K"] / (3 * np.pi * eta_Pa_s * diameter_m) * 1e18

D0_nm2_s = diffusion_nm2_s(SYSTEM)
t_B_s = SYSTEM["diameter_nm"]**2 / D0_nm2_s
print(f"d₀={SYSTEM['diameter_nm']:g} нм, ζ={SYSTEM['zeta_mV']:g} мВ, "
      f"x_EtOH={SYSTEM['ethanol_volume_fraction']:.2f}, T={SYSTEM['temperature_K']:.2f} K")
print(f"Введено η={SYSTEM['viscosity_mPa_s']:g} мПа·с; "
      f"D₀={D0_nm2_s/1e6:.3f} мкм²/с; d₀²/D₀={t_B_s:.3g} с")


## 2. Правило движения и концентрирования

Переходим к безразмерным координатам $\mathbf x_i=\mathbf r_i/d_0$ и времени $\tau=tD_0/d_0^2$. Для фиксированного размера частиц на шаге $\Delta\tau$ используем Эйлера–Маруяму:

$$\Delta\mathbf x_i = \frac{D_i}{D_0}\mathbf f_i\Delta\tau
       +\sqrt{2(D_i/D_0)\Delta\tau}\,\boldsymbol\xi_i,
\qquad \mathbf f_i=-\nabla_{\mathbf x_i}\bigl(U/k_BT\bigr),$$

где компоненты $\boldsymbol\xi_i$ независимы и стандартно нормальны. Гидродинамическая связь между частицами отброшена. Перед шагом движения линейный размер периодической площадки $L$ уменьшается, а координаты аффинно масштабируются $\mathbf x\leftarrow\mathbf xL_{\rm new}/L_{\rm old}$. Доля площади дисков $\phi_{2D}=\sum_i\pi(d_i/2)^2/L^2$ возрастает от 0,25 до 0,72.

**Физический смысл заданной усадки:** это эффективное повышение поверхностной концентрации в движущемся *локальном участке* из-за притока/захвата частиц при сушке. У высыхающего слоя на неподвижной подложке площадь всей подложки от испарения не уменьшается. Здесь также нет вычисления скорости испарения по $T$ и составу. Без измеренного поля потока и закона концентрирования время усадки нельзя считать прогнозом времени высыхания.

Для пары $i,j$ при $\sigma_{ij}=(d_i+d_j)/(2d_0)$ и $s=\max(r-\sigma_{ij},0)$ задан игрушечный потенциал:

$$\frac{U_{ij}}{k_BT}=\frac{k}{2}\max\!\left(1-\frac r{\sigma_{ij}},0\right)^2
 + U_{\rm rep}(r)-H\exp\!\left[-\frac{s^2}{2(w\sigma_{ij})^2}\right],$$

где $U_{\rm rep}=A e^{-s/\lambda}$ при $r\ge\sigma_{ij}$ и линейно продолжается при перекрытии: $U_{\rm rep}=A[1+(\sigma_{ij}-r)/\lambda]$. Параметры $k=1000$, $w=0{,}09$, $A$, $H$ измерены в $k_BT$, $\lambda$ — в долях $d_0$. Мягкое ядро допускает небольшие перекрытия: их число надо проверять, а не выдавать модель за точно твёрдые сферы. Короткодействующее притяжение $H$ лишь условно иллюстрирует склонность к агрегации.


In [ ]:
def repulsion_from_zeta(zeta_mV, A_ref_kBT=3.0, zeta_ref_mV=35.0):
    """Только учебная связь при фиксированной среде; НЕ формула DLVO."""
    assert zeta_ref_mV > 0 and A_ref_kBT >= 0
    return A_ref_kBT * (abs(zeta_mV) / zeta_ref_mV)**2

def initialize_positions(d, L, rng):
    """Последовательно разместить частицы с r > 0,72 контактного расстояния."""
    x = np.empty((len(d), 2))
    for i in range(len(d)):
        for attempt in range(10_000):
            trial = rng.uniform(0, L, size=2)
            delta = x[:i] - trial
            delta -= L * np.rint(delta / L)
            distances = np.linalg.norm(delta, axis=1)
            if np.all(distances > 0.72 * (d[:i] + d[i]) / 2):
                x[i] = trial
                break
        else:
            raise RuntimeError("Не удалось разместить частицы: уменьшите начальную долю покрытия")
    return x

def pair_geometry(x, d, L):
    """Периодические минимальные образы; d, x, L в единицах d₀."""
    delta = x[:, None, :] - x[None, :, :]
    delta -= L * np.rint(delta / L)
    r = np.linalg.norm(delta, axis=-1)
    np.fill_diagonal(r, 1e6)  # конечное число предотвращает неопределённость inf*0
    sigma = (d[:, None] + d[None, :]) / 2
    return delta, r, sigma

def pair_forces(x, d, L, A, interaction_range, adhesion, width=0.09, stiffness=1000.):
    """Сумма сил -grad(U/kBT); длина и силы безразмерны."""
    delta, r, sigma = pair_geometry(x, d, L)
    gap = r - sigma
    s = np.maximum(gap, 0.)
    core = stiffness * np.maximum(-gap, 0.) / sigma**2
    repulsion = A / interaction_range * np.exp(-np.minimum(s / interaction_range, 100))
    attraction = -(adhesion / width**2) * (s / sigma**2) * np.exp(-0.5 * (s / (width * sigma))**2)
    radial_force = core + repulsion + attraction
    np.fill_diagonal(radial_force, 0.)
    return np.sum(radial_force[:, :, None] * delta / r[:, :, None], axis=1)

def simulate_layer(seed, scenario, system=SYSTEM, n=64, phi_initial=.25, phi_final=.72, dt=.0005):
    rng = np.random.default_rng(seed)
    assert system["diameter_nm"] > 0 and system["viscosity_mPa_s"] > 0
    assert 0 < phi_initial < phi_final < np.pi / (2 * np.sqrt(3))
    assert 0 <= scenario["size_cv"] < .3 and scenario["interaction_range"] > 0
    assert scenario["compression_steps"] > 0 and scenario["relaxation_steps"] >= 0
    # Усечённый нормальный закон размеров: CV — вход сценария, не оценка реальной партии.
    d = np.clip(rng.normal(1, scenario["size_cv"], size=n), .60, 1.40)
    area_disks = np.pi * np.sum((d/2)**2)
    L0, L1 = np.sqrt(area_disks / phi_initial), np.sqrt(area_disks / phi_final)
    x = initialize_positions(d, L0, rng)
    L = L0
    D_ratio = 1 / d
    A = repulsion_from_zeta(scenario["zeta_mV"])
    for j in range(scenario["compression_steps"] + scenario["relaxation_steps"]):
        if j < scenario["compression_steps"]:
            L_new = L0 + (L1 - L0) * (j+1) / scenario["compression_steps"]
            x *= L_new / L
            L = L_new
        f = pair_forces(x, d, L, A, scenario["interaction_range"], scenario["adhesion_kBT"])
        x = (x + dt * D_ratio[:, None] * f
             + np.sqrt(2 * dt * D_ratio[:, None]) * rng.normal(size=(n, 2))) % L
    assert np.isfinite(x).all()
    time_unit_s = system["diameter_nm"]**2 / diffusion_nm2_s(system)
    return {"x": x, "diameters": d, "L": L, "seed": seed, "A_kBT": A,
            "scenario": scenario, "dimensionless_time":
            (scenario["compression_steps"] + scenario["relaxation_steps"]) * dt,
            "time_unit_s": time_unit_s}

# Предельная проверка шума свободной частицы в 2D: E[|Δx|²] = 4 D Δτ.
check_rng = np.random.default_rng(7)
check_dt = .2
free_steps = np.sqrt(2 * check_dt) * check_rng.normal(size=(20_000, 2))
assert abs(np.mean(np.sum(free_steps**2, axis=1)) / (4 * check_dt) - 1) < .04
# Внутренние парные силы в периодической области должны компенсироваться.
check_d = np.ones(4)
check_x = np.array([[0.1, .2], [.8, .2], [1.7, 1.2], [2.1, 2.0]])
assert np.allclose(pair_forces(check_x, check_d, 4., 2., .08, .5).sum(axis=0), 0, atol=1e-10)
print("Проверки шума и баланса внутренних сил пройдены")


## 3. Два сценария, три случайные реализации каждого

Сценарии **различаются сразу несколькими** факторами, поэтому разность результата нельзя приписать одному $\zeta$ или скорости концентрирования. Отдельная однофакторная проверка $\zeta$ приведена ниже. Начальные координаты и размеры независимо выбираются при каждом `seed`; это численные повторы, а не независимые синтезы.

| Вход | Медленнее, уже распределение | Быстрее, шире распределение |
|---|---:|---:|
| $\zeta$, мВ | −35 | −10 |
| $A(\zeta)$, $k_BT$ | 3,00 | 0,245 |
| CV диаметра | 0,02 | 0,18 |
| $\lambda/d_0$ | 0,07 | 0,03 |
| $H$, $k_BT$ | 0,1 | 1,5 |
| Шагов сжатия + релаксации | 2200 + 800 | 600 + 400 |

В обоих случаях $N=64$, $\phi_{2D}=0{,}25\to0{,}72$, $\Delta\tau=0{,}0005$. Время в секундах из единицы $d_0^2/D_0$ здесь имеет только смысл **времени заданного численного сценария**, не предсказанного времени испарения.


In [ ]:
SCENARIOS = {
    "Упорядочение": dict(zeta_mV=SYSTEM["zeta_mV"], size_cv=.02, interaction_range=.07,
                         adhesion_kBT=.1, compression_steps=2200, relaxation_steps=800),
    "Неупорядоченная упаковка": dict(zeta_mV=-10., size_cv=.18, interaction_range=.03,
                                     adhesion_kBT=1.5, compression_steps=600, relaxation_steps=400),
}
SEEDS = (101, 102, 103)
results = {name: [simulate_layer(seed, scenario) for seed in SEEDS]
           for name, scenario in SCENARIOS.items()}
for name, realizations in results.items():
    first = realizations[0]
    print(f"{name}: ζ={first['scenario']['zeta_mV']:g} мВ, "
          f"A={first['A_kBT']:.3f} kBT, "
          f"заданное численное время {first['dimensionless_time']:.2f}·d₀²/D₀ "
          f"({first['dimensionless_time'] * first['time_unit_s']:.4f} с при входном D₀; "
          "не время экспериментальной сушки); "
          f"{len(realizations)} seed")


## 4. Структурные показатели и численный контроль

Для частицы с соседями в первом слое определим $\psi_{6,i}=n_i^{-1}\sum_j\exp(6\mathrm{i}\theta_{ij})$; первый слой здесь **операционно** задан порогом $r_{ij}/\sigma_{ij}<1{,}35$ и требует не меньше четырёх соседей. Среднее $\langle|\psi_{6,i}|\rangle$ характеризует локальные шестиугольные окружения, $|\langle\psi_{6,i}\rangle|$ — их общую ориентацию (разные домены могут компенсироваться). Доля $n_i=6$ чувствительна к выбранному порогу, поэтому не является строгим подсчётом топологических дефектов.

Для $g(r)$ считаем пары центров в кольцах и нормируем на ожидаемое число пар в идеальном двумерном газе при **той же** площади и числе частиц. Из-за конечного малого участка и трёх seed кривые иллюстративны; сравнивайте форму пиков и не извлекайте из них надёжные периоды реального опала. Контроль мягкого ядра сообщает число пар с $r<0{,}9\sigma_{ij}$ и минимум $r/\sigma$: небольшие перекрытия возможны; сильные указывают на плохой шаг, силу или плотность.


In [ ]:
def structure_metrics(realization):
    x, d, L = realization["x"], realization["diameters"], realization["L"]
    delta, r, sigma = pair_geometry(x, d, L)
    contact_ratio = r / sigma
    coordination = np.sum(contact_ratio < 1.35, axis=1)
    psi = np.zeros(len(d), dtype=complex)
    for i in range(len(d)):
        ids = np.flatnonzero(contact_ratio[i] < 1.35)
        if len(ids) >= 4:
            theta = np.arctan2(delta[i, ids, 1], delta[i, ids, 0])
            psi[i] = np.mean(np.exp(6j * theta))
    pair_values = contact_ratio[np.triu_indices(len(d), k=1)]
    return dict(local_order=np.mean(abs(psi)), global_order=abs(np.mean(psi)),
                six_neighbors=np.mean(coordination == 6), psi=psi,
                overlap90=np.count_nonzero(pair_values < .9),
                min_contact_ratio=np.min(pair_values))

def radial_distribution(realization, edges):
    x, d, L = realization["x"], realization["diameters"], realization["L"]
    _, r, _ = pair_geometry(x, d, L)
    distances = r[np.triu_indices(len(d), k=1)]  # длины уже в d₀
    counts, _ = np.histogram(distances, bins=edges)
    annuli = np.pi * np.diff(edges**2)
    ideal_count = len(d) * (len(d)-1) / 2 * annuli / L**2
    return counts / ideal_count

print("Сценарий | seed | среднее |ψ₆| | глобальное |ψ₆| | доля n=6 | пар r<0,9σ | min(r/σ)")
for name, realizations in results.items():
    for realization in realizations:
        m = structure_metrics(realization)
        # Сильных перекрытий быть не должно при выбранных параметрах и seed.
        assert m["min_contact_ratio"] > .8
        print(f"{name} | {realization['seed']} | {m['local_order']:.3f} | "
              f"{m['global_order']:.3f} | {m['six_neighbors']:.3f} | "
              f"{m['overlap90']} | {m['min_contact_ratio']:.3f}")


In [ ]:
from matplotlib.patches import Circle

fig, axes = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
for ax, (name, realizations) in zip(axes, results.items()):
    state = realizations[0]
    x, d, L = state["x"], state["diameters"], state["L"]
    order = np.abs(structure_metrics(state)["psi"])
    for center, diameter, local in zip(x, d, order):
        ax.add_patch(Circle(center, diameter/2, facecolor=plt.cm.viridis(local),
                            edgecolor="0.2", lw=.4))
    ax.set(xlim=(0, L), ylim=(0, L), aspect="equal", xlabel="x/d₀", ylabel="y/d₀",
           title=f"{name}, seed {state['seed']}")
sm = plt.cm.ScalarMappable(norm=plt.Normalize(0, 1), cmap="viridis")
fig.colorbar(sm, ax=axes, label="локальное |ψ₆|", shrink=.8)
plt.show()

edges = np.linspace(0, 3.5, 71)  # верхняя граница < L/2 для каждого сценария
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
for name, realizations in results.items():
    assert all(edges[-1] < state["L"]/2 for state in realizations)
    curves = np.array([radial_distribution(state, edges) for state in realizations])
    mean, sd = curves.mean(axis=0), curves.std(axis=0, ddof=1)
    centers = (edges[1:] + edges[:-1])/2
    line, = ax.plot(centers, mean, label=name)
    ax.fill_between(centers, np.maximum(mean-sd, 0), mean+sd,
                    color=line.get_color(), alpha=.15)
ax.set(xlim=(.7, 3.5), xlabel="r/d₀", ylabel="g(r)",
       title="Парная корреляция, среднее ± SD по численным seed")
ax.legend()
plt.show()


## 5. Меняем только ζ в *условном* отображении

Следующий опыт держит CV размеров, $\lambda$, адгезию и расписание сжатия постоянными. Меняется только входной $\zeta$ и через объявленную связь — $A(\zeta)$. Два seed на точку показывают зависимость от случайной реализации; их разброс не является экспериментальной неопределённостью. Если построить другую, откалиброванную связь $A(\zeta,\mathrm{pH},x_{\rm EtOH},I)$, количественный результат изменится.


In [ ]:
ZETA_SWEEP = (-10., -20., -35., -50.)
SWEEP_SEEDS = (101, 102)
sweep_base = dict(size_cv=.08, interaction_range=.06, adhesion_kBT=.8,
                  compression_steps=900, relaxation_steps=450)
sweep_local = []
for zeta in ZETA_SWEEP:
    scenario = dict(sweep_base, zeta_mV=zeta)
    values = [structure_metrics(simulate_layer(seed, scenario))["local_order"]
              for seed in SWEEP_SEEDS]
    sweep_local.append(values)
sweep_local = np.array(sweep_local)
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.errorbar(np.abs(ZETA_SWEEP), sweep_local.mean(axis=1),
            yerr=sweep_local.std(axis=1, ddof=1), fmt="o-", capsize=3)
ax.set(xlabel="|ζ|, мВ (условно изменяет A при остальных фиксированных входах)",
       ylabel="среднее локальное |ψ₆|", ylim=(0, 1),
       title="Сценарная чувствительность, среднее ± SD по двум seed")
plt.show()
print("|ζ|, мВ; A/kBT; локальный порядок по двум seed")
for zeta, values in zip(ZETA_SWEEP, sweep_local):
    print(f"{abs(zeta):g}; {repulsion_from_zeta(zeta):.3f}; "
          f"{values[0]:.3f}, {values[1]:.3f}")


## 6. Дополнение: заданный перенос к краю капли

Это **другая** задача, не продолжение расчёта структуры. В плоской проекции закреплённой капли радиуса $R=10$ мкм задаём расходящееся поле $\mathbf u(\mathbf r)=u_0\mathbf r/R$. Частицы диффундируют с $D_0$, а при пересечении $r\ge R-d_0/2$ считаются осевшими у края (поглощающая граница). Сравним $u_0=0$ с $u_0=0{,}7$ мкм/с при одинаковых начальных координатах и случайных числах. Эта искусственная скорость **не следует** из состава вода–этанол или скорости испарения: реальная смесь может иметь нестационарный поток, вихри Марангони, меняющийся контактный угол и пространственно зависимое отложение. Модель показывает лишь конкуренцию заданной конвекции и диффузии, без взаимодействия частиц и без вывода о структуре опала.

Здесь $\mathrm{Pe}=u_0R/D_0$ сравнивает время переноса на масштабе капли с диффузией. Частицы, достигшие края между дискретными отсчётами, обнаруживаются лишь на конце шага: при использовании модели для количественного времени осаждения надо изучить чувствительность к $\Delta t$.


In [ ]:
def drop_transport(u0_um_s, seed=345, n=2000, R_um=10., duration_s=6., dt_s=.005):
    rng = np.random.default_rng(seed)
    D_um2_s = diffusion_nm2_s(SYSTEM) / 1e6
    edge_um = R_um - SYSTEM["diameter_nm"] * 1e-3 / 2
    radius = np.sqrt(rng.uniform(size=n)) * edge_um
    angle = rng.uniform(0, 2*np.pi, size=n)
    x = radius[:, None] * np.column_stack((np.cos(angle), np.sin(angle)))
    initial_radius = radius.copy()
    deposited = np.zeros(n, dtype=bool)
    count = [0]
    nsteps = int(round(duration_s / dt_s))
    for _ in range(nsteps):
        noise = rng.normal(size=(n, 2))  # все индексы сохраняют общий шум между сценариями
        dx = (u0_um_s * x / R_um) * dt_s + np.sqrt(2*D_um2_s*dt_s)*noise
        x[~deposited] += dx[~deposited]
        deposited |= np.linalg.norm(x, axis=1) >= edge_um
        count.append(np.count_nonzero(deposited))
    return np.linspace(0, duration_s, nsteps+1), np.array(count)/n, initial_radius, deposited

drop_no_flow = drop_transport(0.)
drop_flow = drop_transport(.7)
Pe_drop = .7 * 10 / (D0_nm2_s/1e6)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for label, out in [("Только диффузия", drop_no_flow), ("Диффузия + заданный поток", drop_flow)]:
    axes[0].plot(out[0], out[1], label=label)
    edges = np.linspace(0, 10, 12)
    total, _ = np.histogram(out[2], bins=edges)
    deposited, _ = np.histogram(out[2][out[3]], bins=edges)
    axes[1].plot((edges[:-1]+edges[1:])/2, deposited/np.maximum(total, 1), "o-", label=label)
axes[0].set(xlabel="Время заданного сценария, с", ylabel="Доля осевших у края")
axes[1].set(xlabel="Начальный радиус, мкм", ylabel="Доля осевших за 6 с", ylim=(0, 1))
axes[0].legend()
axes[1].legend(fontsize=8)
print(f"Заданный сценарий: Pe=u₀R/D₀={Pe_drop:.2f}; "
      f"доля у края без потока {drop_no_flow[1][-1]:.3f}, с потоком {drop_flow[1][-1]:.3f}")
plt.show()


## Вопросы для самостоятельной работы и проверки

1. Сохраните сценарий и seed; уменьшите $\Delta\tau$ вдвое, удвоив число шагов и оставив полное безразмерное время и график $L(\tau)$ теми же. Сравните *ансамбли* структурных метрик, а не отдельные траектории: шум при смене шага переставляет частицы.
2. В отдельном однофакторном расчёте измените только CV размеров, затем только время сжатия. Проверьте несколько seed. Почему два крайних сценария выше не доказывают причинное влияние конкретно $\zeta$?
3. При изменении состава смеси в реальном опыте какие **измеренные** функции и дополнительные параметры потребуются, чтобы заменить постоянные $\eta$, $\zeta$, $A$, $\lambda$, $H$ и предписанный поток? Какие наблюдаемые величины (микроскопия поверхности, толщинный профиль, динамика пятна) можно использовать для независимой проверки?
4. Для капли сравните долю осевших при $\Delta t$ и $\Delta t/2$ при том же времени и нескольких seed. Получаете ли вы из этой модели распределение частиц внутри сформировавшегося осадка? Обоснуйте ответ.

**Источники для расширения модели:** [первичная работа об агрегации и скорости сушки SiO₂, *Soft Matter* (2021)](https://pubs.rsc.org/en/content/articlehtml/2021/sm/d0sm00723d); [эксперимент о влиянии растворителя на сборку SiO₂, *Langmuir* (2023)](https://pubs.acs.org/doi/10.1021/acs.langmuir.2c02890); [броуновская динамика высыхающих коллоидных плёнок, *Soft Matter* (2017)](https://authors.library.caltech.edu/records/emx3b-61457); [наблюдение высыхания капель SiO₂ в воде–этаноле, *ACS Applied Materials & Interfaces* (2019)](https://pubs.acs.org/doi/10.1021/acsami.8b21731). Реальные составы и режимы этих исследований не являются параметрами приведённого учебного примера.
